In [1]:
!nvidia-smi    # should show a GPU (e.g. Tesla T4). If error: GPU is not enabled.
from google.colab import drive
drive.mount('/content/drive')

Sat Oct 24 18:46:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, random, time, copy, glob, shutil
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = '/content/drive/MyDrive/glaucoma'
RAW  = f'{ROOT}/raw'
DATA = f'{ROOT}/data'
OUT  = f'{ROOT}/results'
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
IMG, BATCH, EPOCHS, LR, SEED = 224, 32, 15, 1e-4, 42   # SAME for every model (fair comparison)
# SEED is 42 for the first run; later set SEED = 7, then SEED = 123 and re-run the Day 4 cell
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = (DEVICE == 'cuda')                            # mixed precision = faster training
print('Using device:', DEVICE)
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

Using device: cuda


In [3]:
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])   # ImageNet statistics
train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)), transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(), norm])
eval_tf = transforms.Compose([transforms.Resize((IMG, IMG)), transforms.ToTensor(), norm])
class FundusDS(Dataset):
    def __init__(self, frame, tf):
        self.paths = frame.path.tolist()
        self.labels = frame.label.values.astype('float32')
        self.tf = tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.tf(img), torch.tensor(self.labels[i])

In [4]:
import pandas as pd
df = pd.read_csv(f'{ROOT}/master.csv')
print(df.groupby(['dataset', 'label']).size().unstack())

label         0    1
dataset             
ACRIMA      309  396
G1020       724  296
ORIGA       482  168
RIM-ONE_DL  313  172


In [5]:
def split_source(d, seed):
    trval, te = train_test_split(d, test_size=0.15, stratify=d.label, random_state=seed)
    tr, va = train_test_split(trval, test_size=0.15 / 0.85, stratify=trval.label, random_state=seed)
    return tr, va, te
# quick check
d = df[df.dataset == df.dataset.unique()[0]]
tr, va, te = split_source(d, SEED)
print(len(tr), len(va), len(te), '| glaucoma ratio:',
      tr.label.mean().round(2), va.label.mean().round(2), te.label.mean().round(2))

493 106 106 | glaucoma ratio: 0.56 0.56 0.57


In [6]:
def make_model(name):
    if name == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == 'densenet121':
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, 1)
    elif name == 'mobilenet_v2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == 'efficientnet_b0':
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == 'inception_v3':
        m = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
        m.aux_logits = False; m.AuxLogits = None   # turn off the auxiliary head (simplifies training)
        m.fc = nn.Linear(m.fc.in_features, 1)
    else:
        raise ValueError(name)
    return m.to(DEVICE)
print(sum(p.numel() for p in make_model('mobilenet_v2').parameters()) / 1e6, 'million parameters')

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 159MB/s]


2.225153 million parameters


In [7]:
def predict(model, loader):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                out = model(x)
            ps.append(torch.sigmoid(out.float()).squeeze(1).cpu().numpy())
            ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)
def metrics(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(auc=roc_auc_score(y, p),
                sens=tp / max(tp + fn, 1), spec=tn / max(tn + fp, 1),
                acc=(tp + tn) / len(y))
def make_loader(frame, tf, shuffle=False, drop_last=False):
    return DataLoader(FundusDS(frame, tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=2, drop_last=drop_last)

In [8]:
# ===== N1. Normalization experiment: configuration and checks =====
import os, time, copy
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import cv2                                    # preinstalled on Colab
# ---- Fixed in advance. Do NOT add or remove methods after seeing results. ---------
NORMS    = ['grayworld', 'ben', 'clahe', 'lab_ref', 'colorjit']
MODELS_N = ['mobilenet_v2', 'efficientnet_b0','resnet50','densenet121','inception_v3']      # chosen a priori (largest drop / best cross-source)
# For all five models use: ['resnet50','densenet121','mobilenet_v2','efficientnet_b0','inception_v3']
SEEDS    = [42, 7, 123]
SOURCES  = ['ACRIMA', 'ORIGA', 'RIM-ONE_DL']        # training sources
TARGETS  = SOURCES + ['G1020']                      # G1020 stays test-only
# ---- Settings that must come from the original configuration cell  ---------
for need in ['ROOT', 'IMG', 'BATCH', 'EPOCHS', 'LR', 'DEVICE', 'USE_AMP', 'OUT', 'norm',
             'split_source', 'make_model', 'predict', 'metrics', 'make_loader', 'seed_everything']:
    assert need in globals(), f'missing from the notebook: {need}  (run the original cells first)'
# ---- SAME master table as the baseline run (same row order -> same splits) ---
df = pd.read_csv(f'{ROOT}/master.csv')
assert os.path.exists(df.path.iloc[0]), ('Image paths in master.csv do not exist in this session. '
                                         'Mount Drive / restore the local data copy first.')
assert set(TARGETS) <= set(df.dataset.unique()), 'a dataset is missing from master.csv'
print('IMG', IMG, '| BATCH', BATCH, '| EPOCHS', EPOCHS, '| LR', LR, '| device', DEVICE)
print('conditions:', NORMS, '| models:', MODELS_N, '| seeds:', SEEDS)
print('training runs:', len(NORMS) * len(MODELS_N) * len(SOURCES) * len(SEEDS))

IMG 224 | BATCH 32 | EPOCHS 15 | LR 0.0001 | device cuda
conditions: ['grayworld', 'ben', 'clahe', 'lab_ref', 'colorjit'] | models: ['mobilenet_v2', 'efficientnet_b0', 'resnet50', 'densenet121', 'inception_v3'] | seeds: [42, 7, 123]
training runs: 225


In [9]:
# ===== N2. Normalization methods and transform builder =====
def _mask(a):
    """Pixels that belong to the retina (not the black camera border). Falls back to all pixels."""
    m = a.max(axis=2) > 15
    return m if m.mean() > 0.10 else np.ones(a.shape[:2], dtype=bool)
class ColorNorm:
    """Per-image colour/illumination normalization applied to the already resized PIL image.
    kind: 'none' | 'colorjit' (no normalization) | 'grayworld' | 'ben' | 'clahe' | 'lab_ref'
    ref : (mean(3), std(3)) in LAB, computed from the SOURCE TRAINING split only (lab_ref)."""
    def __init__(self, kind, ref=None):
        self.kind, self.ref = kind, ref
        if kind == 'lab_ref':
            assert ref is not None, 'lab_ref needs reference statistics from the source train split'
    def __call__(self, img):
        if self.kind in ('none', 'colorjit'):
            return img
        a = np.asarray(img.convert('RGB'), dtype=np.float32)
        if self.kind == 'grayworld':                 # equalise the mean of R, G, B (retina pixels)
            m = a[_mask(a)].mean(0)
            a = a * (m.mean() / (m + 1e-6))
        elif self.kind == 'ben':                     # Ben Graham: subtract local average colour
            blur = cv2.GaussianBlur(a, (0, 0), IMG / 30)
            a = 4 * a - 4 * blur + 128
        elif self.kind == 'clahe':                   # local contrast on the lightness channel only
            lab = cv2.cvtColor(a.clip(0, 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
            lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
            a = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)
        elif self.kind == 'lab_ref':                 # per-image LAB mean/std -> fixed source reference
            rgb = a.clip(0, 255).astype(np.uint8)
            lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
            px = lab[_mask(a)]
            m, s = px.mean(0), px.std(0) + 1e-6
            lab = (lab - m) / s * self.ref[1] + self.ref[0]
            a = cv2.cvtColor(lab.clip(0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB).astype(np.float32)
        else:
            raise ValueError(self.kind)
        return Image.fromarray(a.clip(0, 255).astype(np.uint8))
def compute_lab_ref(frame):
    """Reference LAB statistics = average of the per-image LAB mean and std over the SOURCE TRAIN
    images only. Validation, test and every other dataset are never used."""
    rs = transforms.Resize((IMG, IMG)); ms, ss = [], []
    for p in frame.path:
        a = np.asarray(rs(Image.open(p).convert('RGB')), dtype=np.uint8)
        lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB).astype(np.float32)
        px = lab[_mask(a.astype(np.float32))]
        ms.append(px.mean(0)); ss.append(px.std(0))
    return np.mean(ms, 0), np.mean(ss, 0)
def build_tfs(kind, ref=None):
    """Same pipeline as the baseline (Step 3.1); only ColorNorm is inserted after Resize.
    'colorjit' keeps the pipeline but uses a stronger ColorJitter in training only."""
    cn  = ColorNorm(kind, ref)
    jit = (transforms.ColorJitter(0.4, 0.4, 0.4, 0.1) if kind == 'colorjit'
           else transforms.ColorJitter(0.2, 0.2))          # baseline value
    tr = transforms.Compose([transforms.Resize((IMG, IMG)), cn, transforms.RandomHorizontalFlip(),
                             transforms.RandomRotation(15), jit, transforms.ToTensor(), norm])
    ev = transforms.Compose([transforms.Resize((IMG, IMG)), cn, transforms.ToTensor(), norm])
    return tr, ev

In [10]:
# ===== N3. LOOK at the normalized images before training =====
def show_norm_grid():
    kinds = ['none', 'grayworld', 'ben', 'clahe', 'lab_ref']
    ref = compute_lab_ref(split_source(df[df.dataset == 'ACRIMA'], 42)[0])   # demo reference
    names = ['ACRIMA', 'ORIGA', 'RIM-ONE_DL', 'G1020']
    rs = transforms.Resize((IMG, IMG))
    fig, ax = plt.subplots(len(names), len(kinds), figsize=(3 * len(kinds), 3 * len(names)))
    for i, ds in enumerate(names):
        r = df[df.dataset == ds].sample(1, random_state=1).iloc[0]
        base = rs(Image.open(r.path).convert('RGB'))
        for j, k in enumerate(kinds):
            ax[i, j].imshow(ColorNorm(k, ref)(base)); ax[i, j].axis('off')
            ax[i, j].set_title(f'{ds} | {k}', fontsize=8)
    plt.tight_layout(); plt.savefig(f'{OUT}/norm_examples.png', dpi=130); plt.show()
show_norm_grid()
# Check: no image is black, washed out or saturated; the optic disc is still visible in every column.

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# ===== N4. Training function  =====
def train_model_n(name, tr, va, seed, tr_tf, ev_tf, epochs=EPOCHS, patience=5):
    seed_everything(seed)
    model = make_model(name)
    tl = make_loader(tr, tr_tf, shuffle=True, drop_last=True)
    vl = make_loader(va, ev_tf)
    pos = float(tr.label.sum()); neg = len(tr) - pos
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg / pos], device=DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
    best_auc, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x, y in tl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=USE_AMP):
                out = model(x).squeeze(1)
                loss = crit(out.float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(x)
        sched.step()
        yv, pv = predict(model, vl)
        auc = roc_auc_score(yv, pv)
        print(f'   [{name}] epoch {ep+1:02d} train_loss {tot/len(tr):.3f} val_auc {auc:.3f}')
        if auc > best_auc:
            best_auc, best_state, bad = auc, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    yv, pv = predict(model, vl)
    fpr, tpr, th = roc_curve(yv, pv)
    thr = float(th[np.argmax(tpr - fpr)])          # Youden threshold from VALIDATION only
    return model, thr, float(best_auc)

In [ ]:
# ===== N5. Define the runner (resumable; nothing is trained yet) =====
# One CSV per (condition, seed): results_norm_<kind>_seed<seed>.csv, same columns as the baseline
# files plus 'norm'. Finished (model, source) runs are skipped
def run_condition(kind, seed):
    res_file = f'{OUT}/results_norm_{kind}_seed{seed}.csv'
    results = pd.read_csv(res_file).to_dict('records') if os.path.exists(res_file) else []
    done = {(r['model'], r['source']) for r in results}
    for src in SOURCES:
        tr, va, te = split_source(df[df.dataset == src], seed)    # SAME split as the baseline
        ref = compute_lab_ref(tr) if kind == 'lab_ref' else None  # source train images only
        tr_tf, ev_tf = build_tfs(kind, ref)
        for name in MODELS_N:
            if (name, src) in done: continue
            t0 = time.time()
            print(f'=== [{kind}] seed {seed} | train {name} on {src} '
                  f'(train={len(tr)}, val={len(va)}, test={len(te)}) ===')
            model, thr, val_auc = train_model_n(name, tr, va, seed, tr_tf, ev_tf)
            for tgt in TARGETS:
                d_eval = te if tgt == src else df[df.dataset == tgt]
                y, p = predict(model, make_loader(d_eval, ev_tf))
                m = metrics(y, p, thr)
                results.append(dict(norm=kind, seed=seed, model=name, source=src, target=tgt,
                                    setting='in-domain' if tgt == src else 'cross-domain',
                                    n=len(d_eval), val_auc=val_auc, thr=thr, **m))
                print(f'   -> test on {tgt:11s} AUC {m["auc"]:.3f} sens {m["sens"]:.2f} spec {m["spec"]:.2f}')
            pd.DataFrame(results).to_csv(res_file, index=False)    # save after every run
            del model; torch.cuda.empty_cache()
            print(f'   done in {(time.time()-t0)/60:.1f} min\n')

In [ ]:
# ===== Optional sanity check  =====
NORMS_BAK, MODELS_BAK = NORMS, MODELS_N
NORMS, MODELS_N = ['none'], ['mobilenet_v2']
run_condition('none', 42)                      # writes results_norm_none_seed42.csv (not used by N7)
a = pd.read_csv(f'{OUT}/results_norm_none_seed42.csv')
b = pd.read_csv(f'{OUT}/results_seed42.csv')
m = a.merge(b[b.model == 'mobilenet_v2'], on=['model', 'source', 'target'], suffixes=('_new', '_old'))
print(m[['source', 'target', 'auc_new', 'auc_old']].assign(diff=lambda d: d.auc_new - d.auc_old).round(3))
NORMS, MODELS_N = NORMS_BAK, MODELS_BAK

=== [none] seed 42 | train mobilenet_v2 on ACRIMA (train=493, val=106, test=106) ===
   [mobilenet_v2] epoch 01 train_loss 0.306 val_auc 0.997
   [mobilenet_v2] epoch 02 train_loss 0.095 val_auc 1.000
   [mobilenet_v2] epoch 03 train_loss 0.071 val_auc 0.998
   [mobilenet_v2] epoch 04 train_loss 0.023 val_auc 0.998
   [mobilenet_v2] epoch 05 train_loss 0.034 val_auc 0.997
   [mobilenet_v2] epoch 06 train_loss 0.040 val_auc 0.999
   [mobilenet_v2] epoch 07 train_loss 0.018 val_auc 0.999
   -> test on ACRIMA      AUC 0.996 sens 0.95 spec 0.98
   -> test on ORIGA       AUC 0.703 sens 0.57 spec 0.73
   -> test on RIM-ONE_DL  AUC 0.681 sens 0.92 spec 0.22
   -> test on G1020       AUC 0.470 sens 0.74 spec 0.21
   done in 9.0 min

=== [none] seed 42 | train mobilenet_v2 on ORIGA (train=454, val=98, test=98) ===
   [mobilenet_v2] epoch 01 train_loss 0.930 val_auc 0.744
   [mobilenet_v2] epoch 02 train_loss 0.759 val_auc 0.773
   [mobilenet_v2] epoch 03 train_loss 0.677 val_auc 0.774
   [mobil

In [ ]:
# ===== N6. Run the full normalization experiment =====
# Order: one condition at a time (a partial run still has complete conditions). Re-run after a disconnect.
for kind in NORMS:                 # one condition at a time: a partial run still has complete conditions
    for seed in SEEDS:
        run_condition(kind, seed)
print('ALL DONE')

=== [colorjit] seed 7 | train inception_v3 on ACRIMA (train=493, val=106, test=106) ===
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 178MB/s] 


   [inception_v3] epoch 01 train_loss 0.513 val_auc 0.855
   [inception_v3] epoch 02 train_loss 0.327 val_auc 0.963
   [inception_v3] epoch 03 train_loss 0.230 val_auc 0.997
   [inception_v3] epoch 04 train_loss 0.163 val_auc 1.000
   [inception_v3] epoch 05 train_loss 0.175 val_auc 0.999
   [inception_v3] epoch 06 train_loss 0.184 val_auc 0.998
   [inception_v3] epoch 07 train_loss 0.128 val_auc 1.000
   [inception_v3] epoch 08 train_loss 0.124 val_auc 1.000
   [inception_v3] epoch 09 train_loss 0.069 val_auc 1.000
   -> test on ACRIMA      AUC 0.997 sens 0.97 spec 0.98
   -> test on ORIGA       AUC 0.714 sens 0.89 spec 0.35
   -> test on RIM-ONE_DL  AUC 0.800 sens 0.84 spec 0.63
   -> test on G1020       AUC 0.496 sens 0.80 spec 0.22
   done in 4.8 min

=== [colorjit] seed 7 | train inception_v3 on ORIGA (train=454, val=98, test=98) ===
   [inception_v3] epoch 01 train_loss 0.984 val_auc 0.682
   [inception_v3] epoch 02 train_loss 0.893 val_auc 0.767
   [inception_v3] epoch 03 train_